# Multi-Head Attention Lab

Split a residual-stream tensor into heads, apply a causal mask, and verify that output shape and future attention mass are correct.


In [ ]:
import math
import torch

torch.manual_seed(5)

def split_heads(x, n_head):
    B, T, C = x.shape
    if C % n_head != 0:
        raise ValueError("n_embd must be divisible by n_head")
    d = C // n_head
    return x.view(B, T, n_head, d).transpose(1, 2)

def merge_heads(x):
    B, H, T, D = x.shape
    return x.transpose(1, 2).contiguous().view(B, T, H * D)

def causal_attention(x, n_head):
    q = split_heads(x, n_head)
    k = split_heads(x, n_head)
    v = split_heads(x, n_head)
    d = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    T = x.shape[1]
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    out = merge_heads(weights @ v)
    return out, weights

x = torch.randn(2, 5, 12)
out, weights = causal_attention(x, n_head=3)
future = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
future_mass = weights.masked_select(future).abs().sum().item()
assert out.shape == x.shape
assert future_mass == 0.0
assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)))
try:
    causal_attention(torch.randn(1, 3, 10), n_head=3)
    raise AssertionError("expected head divisibility failure")
except ValueError:
    pass
print("PASS: l05-09 multi-head attention checks")
